# Figure1_four_model_benchmark_summary

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'benchmark1'
outdir.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv(model_comparison_dir / 'PCA_CP_VAE_CPVAE_rank2_to_rank10_standardized_model_metrics.csv')
metrics['Model'] = metrics['Model'].replace({'CPVAE':'weighted CPVAE'})
model_order = ['PCA','CP','VAE','weighted CPVAE']
colors = {'PCA':'#4C72B0','CP':'#DD8452','VAE':'#8172B2','weighted CPVAE':'#2A9D8F'}
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), sharex=True)
plot_specs = [('MSE','Reconstruction MSE','MSE'), ('Explained_variance','R² / explained variance','R²')]
for ax, (metric, title, ylabel) in zip(axes, plot_specs):
    for model in model_order:
        sub = metrics[metrics['Model'].eq(model)].sort_values('Rank')
        ax.plot(sub['Rank'], sub[metric], marker='o', ms=3.5, lw=1.7, color=colors[model], label=model)
    ax.set_title(title, fontsize=10, pad=8)
    ax.set_xlabel('Rank / number of programs')
    ax.grid(alpha=0.22)
    ax.set_ylabel(ylabel)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Model', frameon=False, fontsize=8, title_fontsize=8, loc='center left', bbox_to_anchor=(0.875, 0.50))
fig.suptitle('Four-model benchmark summary across ranks 2-10', fontsize=13, y=0.98)
fig.subplots_adjust(top=0.82, bottom=0.17, left=0.08, right=0.84, wspace=0.30)
save_pdf(fig, outdir / 'Figure1_four_model_rank2_to_rank10_benchmark_summary.pdf')
metrics.to_csv(outdir / 'Figure1_four_model_rank2_to_rank10_benchmark_metrics.csv', index=False)

# Rank2-rank10 ROC curve grid for the four-model benchmark.
roc_path = model_comparison_dir / 'PCA_CP_VAE_CPVAE_rank2_to_rank10_reconstruction_ROC_curves.csv'
if roc_path.exists():
    roc = pd.read_csv(roc_path)
    roc['Model'] = roc['Model'].replace({'CPVAE': 'weighted CPVAE'})
    roc = roc[roc['Model'].isin(model_order)].copy()
    roc.to_csv(outdir / 'Figure1_four_model_rank2_to_rank10_reconstruction_ROC_curves.csv', index=False)

    fig, axes = plt.subplots(3, 3, figsize=(10.8, 10.4), sharex=True, sharey=True)
    axes = axes.ravel()
    for ax, rank in zip(axes, range(2, 11)):
        rank_df = roc[roc['Rank'].eq(rank)]
        for model in model_order:
            sub = rank_df[rank_df['Model'].eq(model)].sort_values('FPR')
            if sub.empty:
                continue
            # Keep the PDF light without changing the curve shape in a visible way.
            if len(sub) > 900:
                idx = np.unique(np.linspace(0, len(sub) - 1, 900).astype(int))
                sub_plot = sub.iloc[idx]
            else:
                sub_plot = sub
            auc_value = sub['AUC'].dropna().iloc[0] if sub['AUC'].notna().any() else np.nan
            ax.plot(sub_plot['FPR'], sub_plot['TPR'], lw=1.35, color=colors[model], label=f'{model} ({auc_value:.3f})')
        ax.plot([0, 1], [0, 1], color='0.72', lw=0.8, linestyle='--')
        ax.set_title(f'Rank {rank}', fontsize=9.5, pad=7)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.20, linewidth=0.45)
        ax.tick_params(labelsize=7)
        ax.legend(title='Model (AUC)', frameon=False, fontsize=5.4, title_fontsize=5.9, loc='lower right', borderaxespad=0.08, handlelength=0.9, handletextpad=0.25, labelspacing=0.12)
    for ax in axes[6:]:
        ax.set_xlabel('False positive rate', fontsize=8)
    for ax in axes[::3]:
        ax.set_ylabel('True positive rate', fontsize=8)
    fig.suptitle('Rank2-rank10 reconstruction ROC curves across four models', fontsize=12.5, y=0.975)
    fig.subplots_adjust(top=0.91, bottom=0.07, left=0.075, right=0.98, hspace=0.36, wspace=0.22)
    save_pdf(fig, outdir / 'Figure1_four_model_rank2_to_rank10_reconstruction_ROC_curves.pdf')
else:
    print(f'Missing ROC curve table: {roc_path}')



# Supplement diagnostics copied from driver scripts.
# These outputs collect PCA variance, strongest-defect gene maps, rank8 latent/program
# clusters, and existing VAE/CPVAE training histories into Output/supplement.
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

supp_outdir = base_output_dir / 'supplement'
supp_outdir.mkdir(parents=True, exist_ok=True)
for stale in supp_outdir.glob('*'):
    if stale.is_file():
        stale.unlink()

X_raw_df, X_scaled_df, row_mean, row_std = build_data()
gene_ids = X_scaled_df.index.astype(str).tolist()
ann = load_annotation()
display_map = ann.set_index('locus_ID')['Gene_display'].astype(str).to_dict()

def parse_feature(feature):
    time, space = str(feature).split('_', 1)
    return time, space

def make_gene_summary_for_supplement(X_raw_df):
    records = []
    for gene, row in X_raw_df.iterrows():
        values = row.values.astype(float)
        min_idx = int(np.nanargmin(values))
        max_idx = int(np.nanargmax(values))
        min_feature = X_raw_df.columns[min_idx]
        max_feature = X_raw_df.columns[max_idx]
        min_time, min_space = parse_feature(min_feature)
        max_time, max_space = parse_feature(max_feature)
        records.append({
            'Gene': str(gene),
            'Gene_display': display_map.get(str(gene), str(gene)),
            'mean_fitness': float(np.nanmean(values)),
            'min_fitness': float(np.nanmin(values)),
            'max_fitness': float(np.nanmax(values)),
            'strongest_defect_feature': min_feature,
            'strongest_defect_time': min_time,
            'strongest_defect_space': min_space,
            'strongest_positive_feature': max_feature,
            'strongest_positive_time': max_time,
            'strongest_positive_space': max_space,
            'dynamic_range': float(np.nanmax(values) - np.nanmin(values))
        })
    return pd.DataFrame(records)

gene_summary = make_gene_summary_for_supplement(X_raw_df)
gene_summary.to_csv(supp_outdir / 'Supplement_gene_strongest_defect_summary.csv', index=False)

palette_time = dict(zip(Full_Timepoints, plt.cm.Set2(np.linspace(0, 1, len(Full_Timepoints)))))
palette_space = dict(zip(Spacepoints, plt.cm.tab20(np.linspace(0, 1, len(Spacepoints)))))

def scatter_by_category(df, x, y, category, ordered_categories, palette, title, xlabel, ylabel, outfile, legend_title):
    fig, ax = plt.subplots(figsize=(7.6, 5.0))
    for cat in ordered_categories:
        sub = df[df[category].eq(cat)]
        if sub.empty:
            continue
        ax.scatter(sub[x], sub[y], s=9, alpha=0.72, color=palette.get(cat, '#4C72B0'), label=str(cat), linewidths=0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10.5, pad=9)
    ax.grid(alpha=0.18, linewidth=0.45)
    ax.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=7, title_fontsize=8)
    fig.subplots_adjust(left=0.12, right=0.78, top=0.86, bottom=0.13)
    save_pdf(fig, outfile)

# PCA variance explained and PCA maps.
pca = PCA(n_components=min(8, X_scaled_df.shape[1]), random_state=0)
pca_scores_matrix = pca.fit_transform(X_scaled_df.values.astype(float))
pca_var = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'variance_explained': pca.explained_variance_ratio_,
    'cumulative_variance_explained': np.cumsum(pca.explained_variance_ratio_)
})
pca_var.to_csv(supp_outdir / 'PCA_variance_explained.csv', index=False)
fig, ax = plt.subplots(figsize=(6.2, 3.8))
bars = ax.bar(np.arange(1, len(pca_var)+1), pca_var['variance_explained'] * 100, color='#4C72B0', alpha=0.88)
for bar, value in zip(bars, pca_var['variance_explained'] * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{value:.1f}%', ha='center', va='bottom', fontsize=7)
ax.set_ylim(0, max((pca_var['variance_explained'] * 100).max() * 1.16, 1))
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained (%)')
ax.set_title('PCA variance explained', fontsize=11, pad=8)
ax.set_xticks(np.arange(1, len(pca_var)+1))
ax.grid(axis='y', alpha=0.22)
fig.subplots_adjust(left=0.14, right=0.96, top=0.86, bottom=0.16)
save_pdf(fig, supp_outdir / 'PCA_variance_explained.pdf')

pca_df = pd.DataFrame(pca_scores_matrix[:, :2], columns=['PC1','PC2'])
pca_df['Gene'] = gene_ids
pca_df = pca_df.merge(gene_summary, on='Gene', how='left')
pca_df.to_csv(supp_outdir / 'PCA_rank8_gene_scores_with_strongest_defect.csv', index=False)
scatter_by_category(
    pca_df, 'PC1', 'PC2', 'strongest_defect_space', Spacepoints, palette_space,
    'PCA gene map colored by strongest defect space',
    f"PC1 ({100*pca_var['variance_explained'].iloc[0]:.1f}% variance)",
    f"PC2 ({100*pca_var['variance_explained'].iloc[1]:.1f}% variance)",
    supp_outdir / 'PCA_gene_map_by_strongest_defect_space.pdf',
    'Strongest defect space'
)
scatter_by_category(
    pca_df, 'PC1', 'PC2', 'strongest_defect_time', Full_Timepoints, palette_time,
    'PCA gene map colored by strongest defect time',
    f"PC1 ({100*pca_var['variance_explained'].iloc[0]:.1f}% variance)",
    f"PC2 ({100*pca_var['variance_explained'].iloc[1]:.1f}% variance)",
    supp_outdir / 'PCA_gene_map_by_strongest_defect_time.pdf',
    'Strongest defect time'
)

# Four-model rank8 program-score maps colored by strongest defect time/space.
scores_path = base_output_dir / 'program_landscapes2' / 'Figure2_rank8_gene_program_scores_four_models.csv'
if scores_path.exists():
    scores = pd.read_csv(scores_path)
    scores['Model'] = scores['Model'].replace({'weighted_CPVAE': 'CPVAE', 'weighted CPVAE': 'CPVAE'})
    score_cols = [c for c in scores.columns if c.startswith('Program')]
    rank8_scores = scores.merge(gene_summary, on=['Gene','Gene_display'], how='left')
    rank8_scores.to_csv(supp_outdir / 'Four_model_rank8_gene_program_scores_with_strongest_defect.csv', index=False)
    for model, prefix, title_model in [('CP','CP','CP'), ('VAE','VAE','VAE'), ('CPVAE','CPVAE','CPVAE')]:
        sub = rank8_scores[rank8_scores['Model'].eq(model)].copy()
        if sub.empty or len(score_cols) < 2:
            continue
        x, y = score_cols[0], score_cols[1]
        sub.to_csv(supp_outdir / f'{prefix}_rank8_gene_program_scores_with_strongest_defect.csv', index=False)
        scatter_by_category(
            sub, x, y, 'strongest_defect_space', Spacepoints, palette_space,
            f'{title_model} rank8 gene map colored by strongest defect space',
            f'{title_model} program 1 score', f'{title_model} program 2 score',
            supp_outdir / f'{prefix}_gene_map_by_strongest_defect_space.pdf',
            'Strongest defect space'
        )
        scatter_by_category(
            sub, x, y, 'strongest_defect_time', Full_Timepoints, palette_time,
            f'{title_model} rank8 gene map colored by strongest defect time',
            f'{title_model} program 1 score', f'{title_model} program 2 score',
            supp_outdir / f'{prefix}_gene_map_by_strongest_defect_time.pdf',
            'Strongest defect time'
        )

    # Rank8 clusters / nonlinear programs.
    # CP is clustered in rank8 CP program-score space. VAE uses the original true VAE
    # latent embedding from Module_driver4. CPVAE uses rank8 encoder latent means saved
    # by Figure2 (Program columns are CPVAE encoder mu, renamed here as CPVAE1..CPVAE8).
    def choose_k_and_cluster(values, seed=0):
        rows = []
        max_k = min(10, max(3, values.shape[0] - 1))
        for k in range(2, max_k + 1):
            labels = KMeans(n_clusters=k, random_state=seed, n_init=20).fit_predict(values)
            sil = silhouette_score(values, labels) if len(np.unique(labels)) > 1 else np.nan
            rows.append({'k': k, 'silhouette_score': sil})
        sel = pd.DataFrame(rows)
        best_k = int(sel.sort_values('silhouette_score', ascending=False).iloc[0]['k']) if not sel.empty else 2
        km = KMeans(n_clusters=best_k, random_state=seed, n_init=20)
        labels = km.fit_predict(values).astype(str)
        return best_k, labels, km.cluster_centers_, sel

    def plot_selection(selection, prefix, best_k=None):
        if selection is None or selection.empty:
            return
        fig, ax = plt.subplots(figsize=(5.4, 3.6))
        ax.plot(selection['k'], selection['silhouette_score'], marker='o', lw=1.5, color='#4C72B0')
        if best_k is None and selection['silhouette_score'].notna().any():
            best_k = int(selection.sort_values('silhouette_score', ascending=False).iloc[0]['k'])
        if best_k is not None:
            ax.axvline(best_k, color='#C44E52', lw=1.0, ls='--', label=f'selected k={best_k}')
        ax.set_xlabel('Number of latent clusters')
        ax.set_ylabel('Silhouette score')
        ax.set_title(f'{prefix} latent cluster selection', fontsize=11, pad=8)
        ax.grid(alpha=0.22)
        if best_k is not None:
            ax.legend(frameon=False, fontsize=8)
        fig.subplots_adjust(left=0.16, right=0.96, top=0.86, bottom=0.18)
        save_pdf(fig, supp_outdir / f'{prefix}_latent_cluster_selection_silhouette.pdf')

    def plot_cluster_scatter(sub, prefix, xcol, ycol, cluster_col, centers=None, xlabel=None, ylabel=None, title=None):
        fig, ax = plt.subplots(figsize=(7.2, 5.0))
        cluster_order = sorted(sub[cluster_col].astype(str).unique(), key=lambda v: int(v) if str(v).isdigit() else str(v))
        cmap = plt.cm.tab10(np.linspace(0, 1, max(10, len(cluster_order))))
        for i, cluster in enumerate(cluster_order):
            part = sub[sub[cluster_col].astype(str).eq(str(cluster))]
            ax.scatter(part[xcol], part[ycol], s=12, alpha=0.75, color=cmap[i], label=f'Cluster {cluster}', linewidths=0)
        if centers is not None:
            ax.scatter(centers[:, 0], centers[:, 1], s=110, marker='x', color='black', label='Cluster centers')
        ax.set_xlabel(xlabel or xcol)
        ax.set_ylabel(ylabel or ycol)
        ax.set_title(title or f'{prefix} latent clusters / nonlinear programs', fontsize=11, pad=8)
        ax.grid(alpha=0.18, linewidth=0.45)
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=7, title='Cluster')
        fig.subplots_adjust(left=0.12, right=0.78, top=0.86, bottom=0.13)
        save_pdf(fig, supp_outdir / f'{prefix}_latent_clusters_nonlinear_programs.pdf')

    def save_cluster_summaries_and_heatmaps(sub, prefix, cluster_col):
        cluster_summary = (
            sub.groupby(cluster_col)
            .agg(
                n_genes=('Gene','count'),
                strongest_defect_time_mode=('strongest_defect_time', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
                strongest_defect_space_mode=('strongest_defect_space', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
                mean_fitness_mean=('mean_fitness','mean'),
                min_fitness_mean=('min_fitness','mean'),
                dynamic_range_mean=('dynamic_range','mean')
            )
            .reset_index()
        )
        cluster_summary.to_csv(supp_outdir / f'{prefix}_rank8_latent_cluster_summary.csv', index=False)
        time_cluster = pd.crosstab(sub[cluster_col], sub['strongest_defect_time'], normalize='index').reindex(columns=Full_Timepoints).fillna(0)
        space_cluster = pd.crosstab(sub[cluster_col], sub['strongest_defect_space'], normalize='index').reindex(columns=Spacepoints).fillna(0)
        time_cluster.to_csv(supp_outdir / f'{prefix}_cluster_by_strongest_defect_time_fraction.csv')
        space_cluster.to_csv(supp_outdir / f'{prefix}_cluster_by_strongest_defect_space_fraction.csv')

        fig, ax = plt.subplots(figsize=(6.8, 3.5))
        im = ax.imshow(time_cluster.values, aspect='auto', vmin=0, vmax=max(float(np.nanmax(time_cluster.values)), 1e-9), cmap='viridis')
        ax.set_xticks(range(len(Full_Timepoints)))
        ax.set_xticklabels(Full_Timepoints)
        ax.set_yticks(range(len(time_cluster.index)))
        ax.set_yticklabels(time_cluster.index)
        ax.set_xlabel('Strongest defect time')
        ax.set_ylabel(f'{prefix} cluster')
        ax.set_title(f'{prefix} clusters by defect timing', fontsize=10.5, pad=8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.025)
        cbar.set_label('Fraction of genes in cluster')
        fig.subplots_adjust(left=0.16, right=0.91, top=0.84, bottom=0.18)
        save_pdf(fig, supp_outdir / f'{prefix}_cluster_defect_time_heatmap.pdf')

        fig, ax = plt.subplots(figsize=(8.2, 3.5))
        im = ax.imshow(space_cluster.values, aspect='auto', vmin=0, vmax=max(float(np.nanmax(space_cluster.values)), 1e-9), cmap='magma')
        ax.set_xticks(range(len(Spacepoints)))
        ax.set_xticklabels(Spacepoints, rotation=45, ha='right')
        ax.set_yticks(range(len(space_cluster.index)))
        ax.set_yticklabels(space_cluster.index)
        ax.set_xlabel('Strongest defect location')
        ax.set_ylabel(f'{prefix} cluster')
        ax.set_title(f'{prefix} clusters by GI location', fontsize=10.5, pad=8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.040, pad=0.025)
        cbar.set_label('Fraction of genes in cluster')
        fig.subplots_adjust(left=0.13, right=0.91, top=0.84, bottom=0.25)
        save_pdf(fig, supp_outdir / f'{prefix}_cluster_defect_space_heatmap.pdf')

    # CP rank8 clusters from CP gene program scores.
    cp_sub = rank8_scores[rank8_scores['Model'].eq('CP')].copy()
    if not cp_sub.empty:
        values = cp_sub[score_cols].astype(float).values
        best_k, labels, centers, selection = choose_k_and_cluster(values, seed=42)
        cp_sub['CP_cluster'] = labels
        selection.to_csv(supp_outdir / 'CP_rank8_latent_cluster_selection.csv', index=False)
        cp_sub.to_csv(supp_outdir / 'CP_rank8_latent_gene_embedding_with_clusters.csv', index=False)
        plot_cluster_scatter(
            cp_sub, 'CP', score_cols[0], score_cols[1], 'CP_cluster', centers=centers,
            xlabel='CP program 1 score', ylabel='CP program 2 score',
            title='CP rank8 program clusters / nonlinear programs'
        )
        save_cluster_summaries_and_heatmaps(cp_sub, 'CP', 'CP_cluster')

    # VAE true latent clusters from Module_driver4 outputs.
    vae_latent_path = ai_output_dir / 'VAE_latent_gene_embedding_with_clusters.csv'
    vae_selection_path = ai_output_dir / 'VAE_latent_cluster_selection.csv'
    if vae_latent_path.exists():
        vae_latent = pd.read_csv(vae_latent_path)
        vae_latent['VAE_cluster'] = vae_latent['VAE_cluster'].astype(str)
        vae_latent.to_csv(supp_outdir / 'VAE_latent_gene_embedding_with_clusters.csv', index=False)
        # Also keep the rank8-name file for compatibility, but contents are true VAE latent dimensions.
        vae_latent.to_csv(supp_outdir / 'VAE_rank8_latent_gene_embedding_with_clusters.csv', index=False)
        if vae_selection_path.exists():
            vae_selection = pd.read_csv(vae_selection_path)
            vae_selection.to_csv(supp_outdir / 'VAE_rank8_latent_cluster_selection.csv', index=False)
            plot_selection(vae_selection, 'VAE')
        centers = vae_latent.groupby('VAE_cluster')[['VAE1','VAE2']].mean().values
        plot_cluster_scatter(
            vae_latent, 'VAE', 'VAE1', 'VAE2', 'VAE_cluster', centers=centers,
            xlabel='VAE latent dimension 1', ylabel='VAE latent dimension 2',
            title='VAE latent clusters / nonlinear programs'
        )
        save_cluster_summaries_and_heatmaps(vae_latent, 'VAE', 'VAE_cluster')
    else:
        print(f'Missing true VAE latent embedding: {vae_latent_path}')

    # CPVAE true rank8 encoder latent mean saved directly by Figure2.
    cpvae_latent_path = base_output_dir / 'program_landscapes2' / 'Figure2_rank8_CPVAE_encoder_latent_mean.csv'
    if cpvae_latent_path.exists():
        cpvae_sub = pd.read_csv(cpvae_latent_path)
        cpvae_sub['Gene'] = cpvae_sub['Gene'].astype(str)
        cpvae_sub = cpvae_sub.merge(gene_summary.drop(columns=['Gene_display'], errors='ignore'), on='Gene', how='left')
        latent_cols = [c for c in cpvae_sub.columns if c.startswith('CPVAE')]
        values = cpvae_sub[latent_cols].astype(float).values
        best_k, labels, centers, selection = choose_k_and_cluster(values, seed=42)
        cpvae_sub['CPVAE_cluster'] = labels
        selection.to_csv(supp_outdir / 'CPVAE_rank8_latent_cluster_selection.csv', index=False)
        plot_selection(selection, 'CPVAE', best_k=best_k)
        keep_cols = ['Gene','Gene_display'] + latent_cols + ['mean_fitness','min_fitness','max_fitness','strongest_defect_feature','strongest_defect_time','strongest_defect_space','strongest_positive_feature','strongest_positive_time','strongest_positive_space','dynamic_range','CPVAE_cluster']
        cpvae_sub[keep_cols].to_csv(supp_outdir / 'CPVAE_rank8_latent_gene_embedding_with_clusters.csv', index=False)
        plot_cluster_scatter(
            cpvae_sub, 'CPVAE', 'CPVAE1', 'CPVAE2', 'CPVAE_cluster', centers=centers,
            xlabel='CPVAE latent dimension 1', ylabel='CPVAE latent dimension 2',
            title='CPVAE rank8 latent clusters / nonlinear programs'
        )
        save_cluster_summaries_and_heatmaps(cpvae_sub, 'CPVAE', 'CPVAE_cluster')
    else:
        print(f'Missing true CPVAE rank8 encoder latent mean: {cpvae_latent_path}; run Figure2 first.')
else:
    print(f'Missing rank8 gene-program score table: {scores_path}')

# Existing VAE and CPVAE training-history diagnostics, copied/replotted into supplement.
def plot_training_history(history_path, prefix, outfile_prefix):
    history_path = Path(history_path)
    if not history_path.exists():
        print(f'Missing training history: {history_path}')
        return
    hist = pd.read_csv(history_path)
    hist.to_csv(supp_outdir / f'{outfile_prefix}_training_history.csv', index=False)
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    if 'train_loss' in hist.columns:
        ax.plot(hist['epoch'], hist['train_loss'], lw=1.5, label='train_loss', color='#4C72B0')
    if 'val_loss' in hist.columns:
        ax.plot(hist['epoch'], hist['val_loss'], lw=1.5, label='val_loss', color='#DD8452')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'{prefix} training history', fontsize=11, pad=8)
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, fontsize=8)
    fig.subplots_adjust(left=0.13, right=0.96, top=0.86, bottom=0.16)
    save_pdf(fig, supp_outdir / f'{outfile_prefix}_training_history.pdf')

plot_training_history(ai_output_dir / 'VAE_training_history.csv', 'VAE', 'VAE')
plot_training_history(ai_output_dir / 'CPVAE_rank6_training_history.csv', 'CPVAE', 'CPVAE')

print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/benchmark1
